In [19]:
#Import#

from cellpose import models, core, io, plot
from cellpose import train
from cellpose import metrics
import pickle, numpy as np
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
import os, re
import skimage.filters
from skimage.feature import peak_local_max
from skimage.measure import regionprops
from skimage import exposure, measure, filters, img_as_float
from skimage.measure import regionprops
from skimage.segmentation import watershed
from skimage.segmentation import mark_boundaries
from skimage.filters import sobel
from skimage.feature import blob_dog
from skimage import data, color, filters, segmentation, measure, morphology
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
from scipy.spatial import distance_matrix
from scipy.ndimage import generate_binary_structure, binary_dilation
from scipy.spatial import Delaunay
from scipy.stats import qmc
import pandas as pd
from glob import glob
import cv2
import re
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed

from nucleus_worker import process_single_nucleus

import time

In [20]:
#Data input from the User

folder_path = r'Y:\Group Members\Valentin Aubry\01_Data\Test_Data_Andreas_hard'  # The r in r'Example\Pathway' is very important to get the raw input


# Check if the folder is empty or even exists
if os.path.isdir(folder_path) == True:
    print("Folder Path exists")

    # Getting the list of directories
    dir = os.listdir(folder_path)
    
    # Checking if the list is empty or not
    if len(dir) == 0:
        print("Empty directory")
    else:
        print("Not empty directory")
else:
    print("Folder path not found")

Folder Path exists
Not empty directory


In [21]:
# Data Loading


# High Throughput Data Loading
Cy5_data = glob(os.path.join(folder_path, '*Cy5 SEM.tif'))
DAPI_data = glob(os.path.join(folder_path, '*DAPI SEM.tif'))
DAPI_mask_data = glob(os.path.join(folder_path, '*DAPI SEM_seg.npy'))
FITC_data = glob(os.path.join(folder_path, '*FITC SEM.tif'))
TRITC_data = glob(os.path.join(folder_path, '*TRITC SEM.tif'))


# Sorting the data in case it isn't yet
Cy5_data.sort()
DAPI_data.sort()
DAPI_mask_data.sort()
FITC_data.sort()
TRITC_data.sort()


# Displaying all the data that was loaded
for i in range (len(DAPI_data)):
    print(os.path.basename(Cy5_data[i]))
    print(os.path.basename(DAPI_data[i]))
    print(os.path.basename(DAPI_mask_data[i]))
    print(os.path.basename(FITC_data[i]))
    print(os.path.basename(TRITC_data[i]))





ATR1_24h--W00032--P00015--Z00000--T00000--Cy5 SEM.tif
ATR1_24h--W00032--P00015--Z00000--T00000--DAPI SEM.tif
ATR1_24h--W00032--P00015--Z00000--T00000--DAPI SEM_seg.npy
ATR1_24h--W00032--P00015--Z00000--T00000--FITC SEM.tif
ATR1_24h--W00032--P00015--Z00000--T00000--TRITC SEM.tif
ATR1_8h--W00029--P00035--Z00000--T00000--Cy5 SEM.tif
ATR1_8h--W00029--P00035--Z00000--T00000--DAPI SEM.tif
ATR1_8h--W00029--P00035--Z00000--T00000--DAPI SEM_seg.npy
ATR1_8h--W00029--P00035--Z00000--T00000--FITC SEM.tif
ATR1_8h--W00029--P00035--Z00000--T00000--TRITC SEM.tif
ATR1_UT--W00026--P00005--Z00000--T00000--Cy5 SEM.tif
ATR1_UT--W00026--P00005--Z00000--T00000--DAPI SEM.tif
ATR1_UT--W00026--P00005--Z00000--T00000--DAPI SEM_seg.npy
ATR1_UT--W00026--P00005--Z00000--T00000--FITC SEM.tif
ATR1_UT--W00026--P00005--Z00000--T00000--TRITC SEM.tif
ATR2_24h--W00044--P00021--Z00000--T00000--Cy5 SEM.tif
ATR2_24h--W00044--P00021--Z00000--T00000--DAPI SEM.tif
ATR2_24h--W00044--P00021--Z00000--T00000--DAPI SEM_seg.npy
ATR2_

In [22]:
# Loading Functions

# Function to check if a potential foci is X-times brighter than the immediate Background (preventing false positives)
def local_contrast(image, coordinates, bright_percentage, contrast_threshold=2.5):
    """
    Filter detected foci by comparing their intensity to local background.
    
    This function confirms foci as true positives only if they are significantly
    brighter than their immediate surroundings, reducing false positive detections
    from background noise or imaging artifacts.
    
    Parameters:
    -----------
    image : numpy.ndarray
        2D array of the fluorescence image (grayscale) where foci are detected.
        Typically a single channel (e.g., TRITC, FITC) image.
    
    coordinates : numpy.ndarray
        Array of shape (N, 2) containing (y, x) coordinates of potential foci
        detected by peak_local_max or similar detection methods.
    
    bright_percentage : int or float
        Percentile (0-100) used to define local background brightness.
        For example:
        - 50 compares to median brightness around the focus
        - 25 compares to dimmer regions (more stringent filtering)
        - 75 compares to brighter regions (less stringent filtering)
    
    contrast_threshold : float, optional (default=2.5)
        Minimum contrast ratio between focus and local background.
        Focus intensity must be > (local_background * contrast_threshold).
        Higher values = more stringent filtering (fewer false positives).
        Typical values: 1.5-3.0
    
    Returns:
    --------
    numpy.ndarray
        Array of shape (M, 2) containing confirmed foci coordinates where M <= N.
        Returns empty array of shape (0, 2) if no foci pass the contrast filter.
    
    Notes:
    ------
    - Uses an 11x11 pixel square (±5 pixels) around each potential focus
    - Edge cases are handled by clamping to image boundaries
    - Empty return is shaped (0, 2) for compatibility with downstream processing
    """
    confirmed_coords = []  #Empty array to store the filtered coordinates in later
    for i in coordinates:
        y, x = i   # Get the coordinates for each foci 
        y_min, y_max = max(0, y-6), min(image.shape[0], y+7)  #setting boudaries from y-5 to y+5(Y+6 gets excluded as upper border)
        x_min, x_max = max(0, x-6), min(image.shape[1], x+7)
        square = image[y_min:y_max, x_min:x_max]  #creating a square with the defined boudaries

        potential_foci = image[y, x]
        local_background = np.percentile(square, bright_percentage)  # Selects a pixel with the brightness in the defined percentage for comparison

        if potential_foci > local_background * contrast_threshold:  # If the potential foci is X-times brighter than the selected background, the foci is confirmed
            confirmed_coords.append(i)
            
    # returns the confirmed foci coordinates or if there are none creates an empty array to not crash the display below
    return np.array(confirmed_coords) if confirmed_coords else np.array([]).reshape(0, 2)





def save_dataframe_to_csv(df, folder_path, base_name, data_type):
    """
    Save a DataFrame to CSV with Excel-compatible formatting.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to save
    folder_path : str
        Directory where file will be saved
    base_name : str
        Base filename (without extension)
    data_type : str
        Type of data being saved (e.g., 'foci', 'nucleus', 'cytoplasm')
    """
    filename = f"{folder_path}/{base_name}--{data_type}_DF_SEM.csv"
    
    # Write sep=, line for Excel compatibility
    with open(filename, 'w') as f:
        f.write('sep=,\n')
    
    # Write data with header
    df.to_csv(filename, mode='a', header=True, index=False)


def farthest_point_sampling(points, n_samples):
    """
    Selects a subset of points that are maximally spread apart 
    (farthest-point or maximin sampling).

    Parameters
    ----------
    points : ndarray of shape (N, D)
        The full set of candidate points (e.g., Sobol samples inside a hull).
    n_samples : int
        The number of points to select.

    Returns
    -------
    selected_points : ndarray of shape (n_samples, D)
        The subset of points that are as far apart as possible.

    Algorithm
    ---------
    1. Pick one random starting point.
    2. Compute distances from all points to this selected point.
    3. Iteratively:
       - Choose the point with the largest distance to the nearest selected point.
       - Add it to the selected set.
       - Update distances so that each point stores its distance to the nearest selected point.
    4. Continue until n_samples points are selected.

    This greedy algorithm ensures each new point lies in the largest remaining empty region.
    It produces a very even spatial coverage, similar to Poisson-disk sampling.
    """

    n = len(points)
    if n_samples > n:
        raise ValueError("n_samples cannot exceed number of available points")

    # Picking a random starting point
    selected_idx = [np.random.randint(0, n)]

    # Computing distances from all points to the first selected point
    # Each element in 'dists' stores the distance to the *nearest selected* point so far
    dists = np.linalg.norm(points - points[selected_idx[0]], axis=1)

    # Iteratively adding points farthest from the selected set
    for _ in range(1, n_samples):
        # Find the index of the point that is farthest from all selected points
        next_idx = np.argmax(dists)
        selected_idx.append(next_idx)

        # Update distances: for every point, 
        # store the distance to the nearest selected point (so far)
        new_dists = np.linalg.norm(points - points[next_idx], axis=1)
        dists = np.minimum(dists, new_dists)

    # Returning the selected subset
    return points[selected_idx]


In [23]:
if __name__ == '__main__':    
    
    
    #timer to measure how long it takes
    start_time = time.time()
    
    #-----------Delaunay + Sobel-------------#
    
    #load the convex hull and the bounding box parameters to iterate through the parameters
    with open(r"Y:\Group Members\Valentin Aubry\01_Data\Parameters\Hull\valid_parameter_hull.pkl", "rb") as f:
        delaunay = pickle.load(f)
    param_bounds = dict(np.load(r"Y:\Group Members\Valentin Aubry\01_Data\Parameters\Hull\parameter_bounds.npz"))
    
    #Define function to get an ndarray with the shape of the hull saved (-1 is outside of the hull shape)
    def in_hull(p):
        return delaunay.find_simplex(p) >= 0
    
    
    # Generate stratified parameter samples
    n_parameter_samples = 256  # Target valid samples
    n_sobol = 2048*2*2               # oversampling to ensure we have enough valid ones (increase if needed)
    
    print(f"Generating {n_sobol} Sobol samples...")
    sampler = qmc.Sobol(d=3, scramble=True)
    unit_samples = sampler.random(n=n_sobol)
    
    # Scale Sobol samples from [0,1]³ to your parameter bounds (Bounding box of the selected convex hull)
    scaled_samples = qmc.scale(
        unit_samples,
        l_bounds=[
            param_bounds["bright_pct"][0],
            param_bounds["contrast_thresh"][0],
            param_bounds["percentile_val"][0],
        ],
        u_bounds=[
            param_bounds["bright_pct"][1],
            param_bounds["contrast_thresh"][1],
            param_bounds["percentile_val"][1],
        ],
    )
    
    # Filter all points inside the Delaunay hull
    inside_mask = in_hull(scaled_samples)
    valid_points = scaled_samples[inside_mask]
    
    valid_param_samples = farthest_point_sampling(valid_points, n_samples=n_parameter_samples)
    #To be able to calculate the probability of each foci being detected
    total_iterations = len(valid_param_samples)
    
    print(f"✓ Generated {len(valid_param_samples)} valid parameter combinations")
    print(f"  Bright %: {valid_param_samples[:,0].min():.1f}-{valid_param_samples[:,0].max():.1f}")
    print(f"  Contrast: {valid_param_samples[:,1].min():.2f}-{valid_param_samples[:,1].max():.2f}")
    print(f"  Percentile: {valid_param_samples[:,2].min():.1f}-{valid_param_samples[:,2].max():.1f}")
    
    
    #-------------------------------------#
    
    # CREATE LIST TO ACCUMULATE ALL FOCI DATA FOR ALL IMAGES
    all_foci_data = []
    
    # List to add all Data pertaining the Nuclei
    all_nuclei_data = []
    
    # To store all the results from the parameter iterations
    parameter_results = []
    
    
    
    #High Throughput Data Loading Loop
    for i in range (len(DAPI_data)):
    #for i in range (1,4):
        #Loading the picture on i position from each channel
        Cy5_pic = imageio.imread(Cy5_data[i])
        DAPI_pic = imageio.imread(DAPI_data[i])
        DAPI_mask_pic = np.load(DAPI_mask_data[i], allow_pickle=True)
        FITC_pic = imageio.imread(FITC_data[i])
        TRITC_pic = imageio.imread(TRITC_data[i])
    
        # Extract the name of the picture that will be analyzed to later be able to save it accordingly
        base_name = os.path.basename(DAPI_data[i]).removesuffix('DAPI SEM.tif')
    
        # Extract the well number
        # searching the base name for the string --W and then all the numbers 0-9 with \d and the + to look for one or more of the numbers 0-9 with he () it saves them
        well_match = re.search(r'--W(\d+)', base_name)
        # group(1) gets just the numbers, group(0) would be with the --W as a prefix. If no numbers are found print of Error
        well_number = well_match.group(1) if well_match else None 
    
        # Extract the Posiiton number --> same principle like extraction of well number extraction
        position_match = re.search(r'--P(\d+)', base_name)
        position_number = position_match.group(1) if well_match else None

        #Warning message in case the well number or position number wasn't able to be extracted
        if well_number is None or position_number is None:
        print(f"Warning: missing Well/Position for file {base_name} (well={well_number}, pos={position_number})")
    
        # Converting the TRITC picture to float to make sure that all the pixel values are compatible with eachother
        TRITC_pic = img_as_float(TRITC_pic)
    
        # Extract the actual mask array
        masks = DAPI_mask_pic  # This contains the segmented regions
    
        
        # ------------------- PARALLEL NUCLEUS PROCESSING ------------------- #
        # Prepare list of nuclei (skip label 0 = background)
        cell_numbers = np.unique(masks)[1:]
        #cell_numbers = [np.unique(masks)[42]]
        
        # Build argument tuples for each nucleus to send to workers
        tasks = [
            (cellnum, masks, TRITC_pic, valid_param_samples, total_iterations, well_number, position_number)
            for cellnum in cell_numbers
        ]
        
        # Parallel processing with proper Windows support
        if __name__ == '__main__':  # ✓ CRITICAL for Windows!
            max_workers = min(8, os.cpu_count() - 1) if os.cpu_count() > 1 else 1
            print(f"🚀 Using {max_workers} parallel workers")
            
            from tqdm import tqdm
            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                futures = {executor.submit(process_single_nucleus, task): task for task in tasks}
                
                for future in tqdm(as_completed(futures), total=len(futures), desc="Processing nuclei"):
                    try:
                        foci_data_list, nuclei_data_list, parameter_results_local = future.result()
                        all_foci_data.extend(foci_data_list)
                        all_nuclei_data.extend(nuclei_data_list)
                        parameter_results.extend(parameter_results_local)
                    except Exception as e:
                        task = futures[future]
                        print(f"❌ Error on nucleus {task[0]}: {e}")
        # ------------------------------------------------------------------- #
    
    
    # After cell loop - convert list of dicts to DataFrame:
    if all_nuclei_data:
        all_nuclei_df = pd.DataFrame(all_nuclei_data)
        save_dataframe_to_csv(all_nuclei_df, folder_path, base_name, 'nuclei_3_param_iteration')
    
    
    # AFTER the cell loop ends:
    if all_foci_data:
        all_foci_df = pd.DataFrame(all_foci_data)
        save_dataframe_to_csv(all_foci_df, folder_path, base_name, 'foci_3_param_iteration')
    
    
    '''    
    
    # Plotting Masks
    fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(36, 24))
    
    # Original DAPI Picture
    axes.imshow(TRITC_pic)
    
    
    #-------Display of the number of Foci on the cells---------#
    # Create a dictionary for easy lookup of min/max per cell
    if all_nuclei_data:
        foci_ranges = {d['cell_num']: (d['foci_min'], d['foci_max']) for d in all_nuclei_data}
    else:
        foci_ranges = {}
    
    
            
    # Adding foci counts to the Individual nuclei
    nums = regionprops(masks)
    for prop in nums:
        # Getting the coordinates of the center of the nuclei
        y, x = prop.centroid
        label = prop.label
        
        # Get min and max for this nucleus
        if label in foci_ranges:
            min_val, max_val = foci_ranges[label]
            
            # Decide color and text based on whether count is stable
            if min_val == max_val:
                # Stable count - show in green
                text = str(min_val)
                color = 'lime'  # Bright green for visibility
            else:
                # Variable count - show range in red
                text = f"{min_val}-{max_val}"
                color = 'red'
        else:
            # No data for this cell
            text = "NaN"
            color = 'yellow'
    
    
        # Adding the text at the location of the centroids
        axes.text(x, y, text, color = color, fontsize = 14, ha = 'center', va = 'center', weight = 'bold')
    
    
    axes.axis('off')
    axes.set_title('TRITC - Foci Detection (Green=Stable, Red=Range)', fontsize = 40)
    
    plt.tight_layout()
    plt.show()
    
    '''
    
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    # Convert to minutes and seconds for easier reading
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print(f"Total runtime: {minutes} minutes and {seconds:.2f} seconds")
    
    '''
    One Nucleus Runtime:
    Before optimization:
    Total runtime: 2 minutes and 17.56 seconds
    After middle part optimization:
    Total runtime: 0 minutes and 4.14 seconds
    '''

Generating 8192 Sobol samples...
✓ Generated 256 valid parameter combinations
  Bright %: 51.0-85.5
  Contrast: 2.00-3.82
  Percentile: 3.8-92.0
🚀 Using 8 parallel workers


Processing nuclei: 100%|███████████████████████████████████████████████████████████████| 49/49 [00:27<00:00,  1.78it/s]


🚀 Using 8 parallel workers


Processing nuclei: 100%|█████████████████████████████████████████████████████████████| 148/148 [01:13<00:00,  2.01it/s]


🚀 Using 8 parallel workers


...

🚀 Using 8 parallel workers


Processing nuclei: 100%|█████████████████████████████████████████████████████████████| 110/110 [00:57<00:00,  1.93it/s]


🚀 Using 8 parallel workers


Processing nuclei: 100%|█████████████████████████████████████████████████████████████| 150/150 [01:13<00:00,  2.04it/s]


Total runtime: 12 minutes and 24.03 seconds
